In [ ]:
# papermill parameters cell — injected at runtime
run_id = ""
data_dir = ""
out_dir = ""
area_um2 = None
thickness_nm = None
device_id = ""
voltage_min_v = None
voltage_max_v = None
compliance_current_a = None

In [ ]:
import glob
import json
import os
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out_path = Path(out_dir)
out_path.mkdir(parents=True, exist_ok=True)
data_path = Path(data_dir)

In [ ]:
# Find VAC .data file
vac_files = list(data_path.glob('VAC_*.data'))
if not vac_files:
    # Try all .data files as fallback
    vac_files = list(data_path.glob('*.data'))
if not vac_files:
    raise FileNotFoundError(f"No VAC .data file found in {data_dir}")

vac_file = vac_files[0]
print(f"Loading: {vac_file}")

# Load CSV — columns: Voltage, Current, Direction, Timestamp, Nruns
df = pd.read_csv(vac_file)
print(f"Loaded {len(df)} rows, columns: {list(df.columns)}")

# Normalise column names
col_map = {c.lower(): c for c in df.columns}
v_col = col_map.get('voltage', df.columns[0])
i_col = col_map.get('current', df.columns[1])

V = df[v_col].to_numpy(dtype=float)
I = df[i_col].to_numpy(dtype=float)

# Try loading meta.json for params
meta_file = Path(str(vac_file).replace('.data', '.meta.json'))
meta = {}
if meta_file.exists():
    meta = json.loads(meta_file.read_text())
    params = meta.get('parameters', {})
    if voltage_min_v is None:
        voltage_min_v = params.get('voltage_min_v')
    if voltage_max_v is None:
        voltage_max_v = params.get('voltage_max_v')
    if compliance_current_a is None:
        compliance_current_a = params.get('compliance_current_a')

print(f"V range: {V.min():.3f} .. {V.max():.3f} V")
print(f"I range: {I.min():.3e} .. {I.max():.3e} A")

In [ ]:
# Compute metrics
I_abs = np.abs(I)
I_max = float(np.nanmax(I_abs))
I_min_nonzero = float(np.nanmin(I_abs[I_abs > 0])) if np.any(I_abs > 0) else float(np.nanmin(I_abs))

# Breakdown / compliance estimate: voltage where |I| first reaches compliance
V_breakdown = None
if compliance_current_a is not None:
    compliance_mask = I_abs >= float(compliance_current_a) * 0.9
    if np.any(compliance_mask):
        idx = int(np.argmax(compliance_mask))
        V_breakdown = float(V[idx])

# If no compliance hit, estimate as V at maximum |I|
if V_breakdown is None:
    V_breakdown = float(V[int(np.argmax(I_abs))])

print(f"I_max={I_max:.3e} A, I_min={I_min_nonzero:.3e} A, V_breakdown~{V_breakdown:.2f} V")

In [ ]:
# Plot 1: I-V linear
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot(V, I * 1e6, 'b-o', markersize=3, linewidth=1.2)
ax.set_xlabel('Voltage (V)')
ax.set_ylabel('Current (µA)')
ax.set_title(f'I-V (linear)\n{device_id or run_id[:16]}')
if V_breakdown is not None:
    ax.axvline(V_breakdown, color='r', linestyle='--', alpha=0.7,
               label=f'V_bd≈{V_breakdown:.2f} V')
    ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
I_abs_plot = np.where(I_abs > 0, I_abs, np.nan)
ax2.semilogy(V, I_abs_plot * 1e6, 'g-o', markersize=3, linewidth=1.2)
ax2.set_xlabel('Voltage (V)')
ax2.set_ylabel('|Current| (µA)')
ax2.set_title(f'|I|-V (log-Y)\n{device_id or run_id[:16]}')
if V_breakdown is not None:
    ax2.axvline(V_breakdown, color='r', linestyle='--', alpha=0.7,
                label=f'V_bd≈{V_breakdown:.2f} V')
    ax2.legend(fontsize=8)
ax2.grid(True, which='both', alpha=0.3)

fig.suptitle(f'IV Breakdown — run {run_id[:20]}', fontsize=11)
fig.tight_layout()

plot_path = out_path / 'iv_breakdown.png'
fig.savefig(str(plot_path), dpi=120, format='png')
plt.close(fig)
print(f"Saved: {plot_path}")

In [ ]:
metrics = {
    'I_max': I_max,
    'I_min': I_min_nonzero,
    'V_breakdown': V_breakdown,
    'compliance_current_a': compliance_current_a,
    'n_points': int(len(V)),
}

metrics_path = out_path / 'metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"Metrics: {metrics}")